# PD14 network write example

Parameterizes and executes a `PD14NetworkWriteTask`, which writes a point-neuron cortical
microcircuit (Potjans & Diesmann 2014) to SONATA files.

This example reproduces the **default configuration** of `microcircuit.sonata_network.SonataNetwork`:
the eight populations `L23E, L23I, L4E, L4I, L5E, L5I, L6E, L6I`, their default neuron counts,
external indegrees, firing rates, initial membrane potentials, layer geometry, the default
connection-probability matrix, and the default thalamic input. The parameters are read from the
`microcircuit` defaults (`default_net_dict` / `default_stim_dict`) and translated into
`PointNeuronPopulation` and `PathwayConnectionProbability` blocks, so the resulting SONATA circuit
matches the model's default network.

In [1]:
from pathlib import Path
import obi_auth

from microcircuit.network_params import default_net_dict
from microcircuit.stimulus_params import default_stim_dict

from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.info import Info
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.blocks.point_neuron_populations.pathway import (
    PathwayConnectionProbability,
)
from obi_one.scientific.blocks.point_neuron_populations.population import (
    PointNeuronPopulation,
    SynapseClass,
)
from obi_one.scientific.tasks.pd14_network_write import PD14NetworkWriteScanConfig

from entitysdk import Client
from obi_notebook.get_projects import get_projects

In [2]:
output_root = Path("../../../../../../obi-output/pd14_network_write")

### Read the default microcircuit parameters

All per-population and global values below come from the `microcircuit` defaults, so the example
stays in sync with the model rather than hard-coding numbers.

In [3]:
population_names = list(default_net_dict["populations"])
num_neurons = default_net_dict["full_num_neurons"]
mean_rates = default_net_dict["full_mean_rates"]
k_ext = default_net_dict["K_ext"]
conn_probs = default_net_dict["conn_probs"]  # conn_probs[target][source]
v0_mean = default_net_dict["neuron_params"]["V0_mean"]["optimized"]
v0_std = default_net_dict["neuron_params"]["V0_std"]["optimized"]
layer_geometry = default_net_dict["layer_geometry"]
conn_probs_th = default_stim_dict["conn_probs_th"]  # thalamus -> population
num_th_neurons = default_stim_dict["num_th_neurons"]

# In the default model, populations alternate excitatory / inhibitory (even index excitatory).
synapse_classes = [
    SynapseClass.EXC if i % 2 == 0 else SynapseClass.INH for i in range(len(population_names))
]
population_names

['L23E', 'L23I', 'L4E', 'L4I', 'L5E', 'L5I', 'L6E', 'L6I']

### Build one `PointNeuronPopulation` block per population

Each block carries everything the exporter sets per population: name, neuron count, synapse class,
external indegree, mean firing rate, spatial extent, initial membrane potential distribution, and
the thalamo-cortical connection probability.

In [4]:
populations = {}
for i, name in enumerate(population_names):
    geometry = layer_geometry[name]
    populations[name] = PointNeuronPopulation(
        population_name=name,
        num_neurons=int(num_neurons[i]),
        synapse_class=synapse_classes[i],
        k_ext=int(k_ext[i]),
        mean_rate=float(mean_rates[i]),
        x_extent=tuple(geometry["x"]),
        y_extent=tuple(geometry["y"]),
        z_extent=tuple(geometry["z"]),
        v0_mean=float(v0_mean[i]),
        v0_std=float(v0_std[i]),
        thalamic_connection_probability=float(conn_probs_th[i]),
    )
populations

{'L23E': PointNeuronPopulation(type='PointNeuronPopulation', population_name='L23E', num_neurons=20683, synapse_class=<SynapseClass.EXC: 'EXC'>, k_ext=1600, mean_rate=0.903, thalamic_connection_probability=0.0, x_extent=(0.0, 100.0), y_extent=(0.0, 100.0), z_extent=(0.0, 100.0), v0_mean=-68.28, v0_std=5.36),
 'L23I': PointNeuronPopulation(type='PointNeuronPopulation', population_name='L23I', num_neurons=5834, synapse_class=<SynapseClass.INH: 'INH'>, k_ext=1500, mean_rate=2.965, thalamic_connection_probability=0.0, x_extent=(0.0, 100.0), y_extent=(0.0, 100.0), z_extent=(0.0, 100.0), v0_mean=-63.16, v0_std=4.57),
 'L4E': PointNeuronPopulation(type='PointNeuronPopulation', population_name='L4E', num_neurons=21915, synapse_class=<SynapseClass.EXC: 'EXC'>, k_ext=2100, mean_rate=4.414, thalamic_connection_probability=0.0983, x_extent=(0.0, 100.0), y_extent=(100.0, 200.0), z_extent=(0.0, 100.0), v0_mean=-63.33, v0_std=4.74),
 'L4I': PointNeuronPopulation(type='PointNeuronPopulation', populati

### Assemble the scan config and add the populations

The global parameters in `Initialize` are left at their defaults (which match the model defaults),
except for enabling the thalamus via `num_th_neurons`. Populations are added first so that each
block obtains a reference (`population.ref`) that the pathway blocks can point to.

In [ ]:
scan_config = PD14NetworkWriteScanConfig.empty_config()
scan_config.set(
    PD14NetworkWriteScanConfig.Initialize(
        num_th_neurons=int(num_th_neurons),
        n_scaling=0.2,
        k_scaling=0.2,
    ),
    name="initialize",
)
scan_config.set(
    Info(
        campaign_name="PD14Network - second test",
        campaign_description="First attempts with scaling factors 0.2"
    ),
    name="info"
)

for name, population in populations.items():
    scan_config.add(population, name=name)

### Add the connection-probability matrix as `PathwayConnectionProbability` blocks

The default `conn_probs` matrix is indexed `conn_probs[target][source]`. We add one pathway block
for every non-zero entry; zero entries are left out and default back to probability 0.0.

In [20]:
for i, target_name in enumerate(population_names):
    for j, source_name in enumerate(population_names):
        probability = float(conn_probs[i][j])
        if probability == 0.0:
            continue
        scan_config.add(
            PathwayConnectionProbability(
                source=populations[source_name].ref,
                target=populations[target_name].ref,
                connection_probability=probability,
            ),
            name=f"{source_name}__{target_name}",
        )

scan_config = scan_config.validated_config()
scan_config

PD14NetworkWriteScanConfig(type='PD14NetworkWriteScanConfig', info=Info(type='Info', campaign_name='PD14Network - second test', campaign_description='First attempts with scaling factors 0.2'), initialize=Initialize(type='PD14NetworkWriteScanConfig.Initialize', n_scaling=0.2, k_scaling=0.2, bg_rate=8.0, psp_exc_mean=0.15, g=-4.0, weight_rel_std=0.1, delay_exc_mean=1.5, delay_inh_mean=0.75, delay_rel_std=0.5, rng_seed=55, num_th_neurons=902, psp_th=0.15, delay_th_mean=1.5, delay_th_rel_std=0.5, population_name='microcircuit', background_name='background', thalamus_name='thalamus', write_indices=True), point_neuron_populations={'L23E': PointNeuronPopulation(type='PointNeuronPopulation', population_name='L23E', num_neurons=20683, synapse_class=<SynapseClass.EXC: 'EXC'>, k_ext=1600, mean_rate=0.903, thalamic_connection_probability=0.0, x_extent=(0.0, 100.0), y_extent=(0.0, 100.0), z_extent=(0.0, 100.0), v0_mean=-68.28, v0_std=5.36), 'L23I': PointNeuronPopulation(type='PointNeuronPopulation'

### Platform authentication - generate Client

In [21]:
token = obi_auth.get_token(environment="staging", auth_mode="daf")
project_context = get_projects(token, env="staging")

Dropdown(description='Select:', options=(('Proximal vs. distal inhibition', {'id': '32008ab5-76cf-47cd-b98b-a5…

### Generate the single-coordinate scan and execute the task

In [24]:
entity_client = Client(token_manager=token, project_context=project_context, environment="staging")

grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()
run_tasks_for_generated_scan(
    grid_scan,
    db_client=entity_client
)

SONATA files will be written to: ../../../../../../obi-output/pd14_network_write/0
  Directory already existed. Old data will be overwritten.

Neuron numbers are scaled by a factor of 0.200.
Indegrees are scaled by a factor of 0.200.
  Weights and DC input are adjusted to compensate.

Creating neuronal populations.
  15435 neurons in node population 'microcircuit'.
Creating virtual populations for background input.
  4137 virtual nodes in node population 'background_L23E'.
  1167 virtual nodes in node population 'background_L23I'.
  4383 virtual nodes in node population 'background_L4E'.
  1096 virtual nodes in node population 'background_L4I'.
  970 virtual nodes in node population 'background_L5E'.
  213 virtual nodes in node population 'background_L5I'.
  2879 virtual nodes in node population 'background_L6E'.
  590 virtual nodes in node population 'background_L6I'.
Creating virtual population for thalamic input.
  902 virtual nodes in node population 'thalamus'.
Connecting neuronal

[None]

### Inspect the written SONATA circuit

In [8]:
import libsonata

instance = grid_scan.single_configs[0]
circuit_root = Path(instance.coordinate_output_root)
print("Written to:", circuit_root)
print("Files:", sorted(p.name for p in circuit_root.iterdir()))

circuit = libsonata.CircuitConfig.from_file(str(circuit_root / "circuit_config.json"))
print("\nNode populations:", sorted(circuit.node_populations))
print("Edge populations:", sorted(circuit.edge_populations))

nodes = circuit.node_population("microcircuit")
print("\nTotal cortical neurons:", nodes.size)
edges = circuit.edge_population("microcircuit__microcircuit__chemical")
print("Total recurrent connections:", edges.size)

Written to: ../../../../../../obi-output/pd14_network_write/0
Files: ['circuit_config.json', 'edges.h5', 'net_dict.json', 'node_sets.json', 'nodes_background_L23E.h5', 'nodes_background_L23I.h5', 'nodes_background_L4E.h5', 'nodes_background_L4I.h5', 'nodes_background_L5E.h5', 'nodes_background_L5I.h5', 'nodes_background_L6E.h5', 'nodes_background_L6I.h5', 'nodes_microcircuit.h5', 'nodes_thalamus.h5', 'obi_one_coordinate.json', 'rng_seed.json', 'sonata_dict.json', 'stim_dict.json']

Node populations: ['background_L23E', 'background_L23I', 'background_L4E', 'background_L4I', 'background_L5E', 'background_L5I', 'background_L6E', 'background_L6I', 'microcircuit', 'thalamus']
Edge populations: ['background_L23E__microcircuit__chemical', 'background_L23I__microcircuit__chemical', 'background_L4E__microcircuit__chemical', 'background_L4I__microcircuit__chemical', 'background_L5E__microcircuit__chemical', 'background_L5I__microcircuit__chemical', 'background_L6E__microcircuit__chemical', 'back